In [1]:
from pathlib import Path
import sys

root = Path.cwd().parent
sys.path.insert(0, str(root))

In [15]:
from qsdl.husimi import apply_husimi
from qsdl.noise import apply_channel_noise
from qsdl.config import XMAX, GRID, CUTOFF
from qsdl.noise import apply_gauss_noise
from qsdl.mle import mle_husimi
import numpy as np
import qutip as qt
import h5py
import json

h5_path = f"{root}/data/train_noisy_35.h5"

In [18]:
with h5py.File(h5_path, 'r') as f:
    rhos = list(f["rhos_clean"])
    metadata = list(f["metadata"])

In [29]:
rng = np.random.default_rng(67)

res = []
for i in range(len(rhos)):
    rho0 = qt.Qobj(rhos[i])

    # wazne, dict wywala przy apply_channel_noise
    meta = metadata[N]
    if isinstance(meta, bytes):
        meta = meta.decode()
    params = json.loads(meta)

    rho = apply_channel_noise(rho0, params)
    rho = rho / rho.tr()
    Q = apply_husimi(rho, GRID, XMAX)
    Q = apply_gauss_noise(Q, params, rng)

    mle_info = mle_husimi(Q, CUTOFF, XMAX, 100)

    fidel = qt.fidelity(qt.Qobj(nw['rho']), rho0)

    res.append({"fidelity": fidel, "mle_info": mle_info})

In [ ]:
res

In [44]:
fids = [i["fidelity"] for i in res]
times = [i["mle_info"]["time_s"] for i in res]
indices = [i for i in range(len(rhos))]

In [48]:
summary = {
    "n": 35,
    "max_iters": 50,
    "mean_fidelity": float(np.mean(fids)),
    "mean_time_s": float(np.mean(times)),
    "per_sample": [
        {"i": int(i), "fidelity": float(f), "time_s": float(t), "iters": 100}
        for i, f, t in zip(indices, fids, times)
    ],
}
with open("noisy_35.json", "w") as f:
    json.dump(summary, f, indent=2)
